# D.R.O.N.A. — Notebook 08: Diffusion Policy Ablation (LeRobot, Colab T4)

**Research Contribution C3 (ablation)** — train a **Diffusion Policy** (Chi et al. 2023) on the *same* gesture `LeRobotDataset` used for ACT in notebook 07, and compare ACT vs Diffusion vs keyframe baseline on smoothness and success.

Diffusion Policy models a multimodal action distribution via conditional denoising. The point of this notebook is the **controlled comparison**: same data, same eval harness (`drona.interaction.sim_eval`), different policy class.

**Runtime:** Colab **T4 GPU**. Diffusion training is heavier than ACT — budget ~20–40 min.

> Run notebook **07 first** (it generates the seed demos and the `LeRobotDataset`).

## 1. Install + clone (skip if 07 already ran in this session)

In [ ]:
%pip install -q "git+https://github.com/huggingface/lerobot.git"
import os
if not os.path.exists("drona"):
    !git clone https://github.com/<your-username>/D.R.O.N.A.git 2>/dev/null || echo "Upload/clone repo, then cd into it."
    %cd D.R.O.N.A
%pip install -q -e . || %pip install -q numpy loguru pydantic

## 2. Rebuild the dataset if needed

If `data/lerobot/drona_gestures` already exists from notebook 07 this is a no-op; otherwise regenerate the seed demos and convert.

In [ ]:
import os, numpy as np
from drona.interaction.demonstration import (
    GESTURE_KEYFRAMES, DemonstrationDataset, DemonstrationEpisode,
    interpolate_keyframes, clamp_joints,
)
from drona.interaction.lerobot_dataset import build_lerobot_dataset

ROOT = "data/lerobot/drona_gestures"
if not os.path.exists(ROOT):
    rng = np.random.default_rng(42)
    ds = DemonstrationDataset(name="drona_gestures"); ep_idx = 0
    for gesture, keyframes in GESTURE_KEYFRAMES.items():
        for _ in range(25):
            traj = interpolate_keyframes(keyframes, dt=0.05)
            ep = DemonstrationEpisode(episode_index=ep_idx, gesture_label=gesture)
            for i, (q, t) in enumerate(traj):
                nq = traj[i + 1][0] if i + 1 < len(traj) else q
                ep.add_frame(
                    obs=clamp_joints(q + rng.normal(0, 0.02, q.shape).astype(np.float32)),
                    action=clamp_joints(nq + rng.normal(0, 0.02, nq.shape).astype(np.float32)),
                    timestamp=t)
            ep.mark_terminal(); ds.add_episode(ep); ep_idx += 1
    build_lerobot_dataset(ds, repo_id="drona/gestures", fps=20, root=ROOT)
print("dataset ready at", ROOT)

## 3. Train Diffusion Policy

In [ ]:
!python -m lerobot.scripts.train \
    --dataset.repo_id=drona/gestures \
    --dataset.root=data/lerobot/drona_gestures \
    --policy.type=diffusion \
    --policy.horizon=16 \
    --policy.n_action_steps=8 \
    --policy.n_obs_steps=2 \
    --batch_size=32 \
    --steps=30000 \
    --output_dir=data/checkpoints/diffusion \
    --job_name=drona_diffusion \
    --device=cuda \
    --wandb.enable=false

## 4. Three-way comparison: keyframe vs ACT vs Diffusion

In [ ]:
import json
from drona.interaction.act_policy import KeyframePolicy, LeRobotACTPolicy
from drona.interaction.diffusion_policy import LeRobotDiffusionPolicy
from drona.interaction.sim_eval import evaluate_policy

def safe_factory(cls, base):
    def f(gesture):
        try:
            return cls(f"{base}/{gesture}", device="cuda")
        except Exception:
            try:
                return cls(base, device="cuda")
            except Exception:
                return KeyframePolicy(gesture)
    return f

reports = {
    "keyframe": evaluate_policy(lambda g: KeyframePolicy(g)),
    "act": evaluate_policy(safe_factory(LeRobotACTPolicy, "data/checkpoints/act")),
    "diffusion": evaluate_policy(safe_factory(LeRobotDiffusionPolicy, "data/checkpoints/diffusion")),
}
summary = {k: {"success_rate": r.success_rate, "mean_jerk": r.mean_jerk,
               "mean_path_length": r.mean_path_length} for k, r in reports.items()}
print(json.dumps(summary, indent=2))

## 5. Record results

Log the success-rate and mean-jerk table for keyframe / ACT / Diffusion in `docs/research_papers.md` and the thesis evaluation chapter. The expected finding for C3: learned policies (ACT, Diffusion) achieve **lower jerk** (smoother, more natural gestures) than the scripted keyframe baseline while maintaining comparable success rate.